<a href="https://colab.research.google.com/github/mrhelek/proyekakhirNLP/blob/main/Skenario_3_bitdistiller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Jun  4 14:56:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Clone repo
!git clone https://github.com/BrownianNotion/BitDistiller.git
%cd BitDistiller

# Install requirement (ini akan downgrade transformer, accelerate, peft)
!pip install -r requirements.txt
!pip install protobuf sentencepiece

Cloning into 'BitDistiller'...
remote: Enumerating objects: 2602, done.
remote: Counting objects: 100% (587/587), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 2602 (delta 493), reused 477 (delta 442), pack-reused 2015 (from 1)
Receiving objects: 100% (2602/2602), 45.01 MiB | 27.18 MiB/s, done.
Resolving deltas: 100% (1190/1190), done.
Filtering content: 100% (2/2), 51.69 MiB | 10.34 MiB/s, done.
/content/BitDistiller
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.2/183.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import login
import os
from google.colab import userdata

# Retrieve the token from Colab's secrets manager
token = userdata.get('HF_TOKEN')
login(token=token)
print("✅ Login berhasil menggunakan token yang disediakan.")

✅ Login berhasil menggunakan token yang disediakan.


In [ ]:
# ========== Cari model BitDistiller yang sudah jadi ==========
from huggingface_hub import list_models

# Coba beberapa kombinasi pencarian
print("Mencari model TinyLlama + BitDistiller publik...")
candidates = []
for query in ["bitdistiller TinyLlama", "TinyLlama 4bit int", "TinyLlama 4-bit"]:
    models = list_models(search=query)
    for m in models:
        if "TinyLlama" in m.modelId and ("4bit" in m.modelId or "bit" in m.modelId.lower()):
            candidates.append(m.modelId)

# Hapus duplikat
candidates = list(set(candidates))
if candidates:
    print("✅ Model kandidat ditemukan:")
    for c in candidates:
        print(c)
    # Gunakan model pertama sebagai model kita
    pretrained_model = candidates[0]
    print(f"\nAkan dievaluasi: {pretrained_model}")
else:
    print("❌ Tidak ada model publik yang cocok. Lanjut ke proses clipping+training...")
    pretrained_model = None  # Nanti kita buat sendiri

Mencari model TinyLlama + BitDistiller publik...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Model kandidat ditemukan:
RichardErkhov/TinyLlama_-_TinyLlama-1.1B-intermediate-step-240k-503b-4bits
RaviKanur/TinyLlama_4bit_quant_v1
RichardErkhov/habanoz_-_TinyLlama-1.1B-intermediate-step-715k-1.5T-lr-5-4epochs-oasst1-top1-instruct-V1-4bits
RichardErkhov/TinyLlama_-_TinyLlama-1.1B-intermediate-step-480k-1T-4bits
fredericowieser/TinyLlama_v1.1_mix_wikitext_alpaca_1bit_BitDistiller_baseline
kaitchup/TinyLlama-1.1B-intermediate-step-1431k-3T-awq-4bit
BrownianNotion/TinyLlama_v1.1_mix_wikitext_alpaca_2bit_BitDistiller_baseline
RichardErkhov/TinyLlama_-_TinyLlama-1.1B-intermediate-step-1431k-3T-4bits
RichardErkhov/invalid-coder_-_TinyLlama-1.1B-intermediate-step-1431k-3T-laser-dpo-4bits
kaitchup/TinyLlama-1.1B-intermediate-step-1431k-3T-gptq-4bit
RichardErkhov/TinyLlama_-_TinyLlama-1.1B-intermediate-step-715k-1.5T-4bits
RichardErkhov/habanoz_-_TinyLlama-1.1B-intermediate-step-715k-1.5T-lr-5-3epochs-oasst1-top1-instruct-V1-4bits
RichardErkhov/TinyLlama_-_TinyLlama-1.1B-intermediate-ste

In [ ]:
# Model BitDistiller 2-bit yang sudah tersedia di HF
model_id = "BrownianNotion/TinyLlama_v1.1_mix_wikitext_alpaca_2bit_BitDistiller_baseline"

# Direktori lokal untuk menyimpan model
import os
local_model_dir = "models/tinyllama_2bit_bitdistiller"
os.makedirs(local_model_dir, exist_ok=True)

# Unduh model
from huggingface_hub import snapshot_download
print(f"Mengunduh model {model_id}...")
snapshot_download(repo_id=model_id, local_dir=local_model_dir)
print("Unduhan selesai.")

Mengunduh model BrownianNotion/TinyLlama_v1.1_mix_wikitext_alpaca_2bit_BitDistiller_baseline...


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

global_step360/mp_rank_00_model_states.p(…):   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

byteorder:   0%|          | 0.00/6.00 [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/6.39k [00:00<?, ?B/s]

training_args/data.pkl:   0%|          | 0.00/5.53k [00:00<?, ?B/s]

version:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

Unduhan selesai.


In [ ]:
%cd /content/BitDistiller/test/general

model_path = "../../models/tinyllama_2bit_bitdistiller"

# === Perplexity (WikiText-2) ===
print("Menghitung Perplexity...")
!python wiki_ppl.py \
    --model {model_path} \
    --quant_type int \
    --bits 2 \
    --group_size 128

# === Zero-shot QA (HellaSwag + PIQA) ===
print("\nMenghitung Zero-shot Accuracy...")
!python llm_eval.py \
    --model ../../models/tinyllama_2bit_bitdistiller \
    --eval_tasks hellaswag \
    --test_set \
    --bits 2 \
    --group_size 128 \
    --quant_type int \
    --num_fewshot 0

/content/BitDistiller/test/general
Menghitung Perplexity...
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Namespace(model='../../models/tinyllama_2bit_bitdistiller', dev='cuda:0', quant_type='int', bits=2, group_size=128)
loading the model...
pseudo int weight quantization...: 100% 22/22 [00:00<00:00, 51.03it/s]
README.md: 10.5kB [00:00, 26.3MB/s]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): 100% 733k/733k [00:00<00:00, 1.33MB/s]
wikitext-2-raw-v1/train-00000-of-00001.p(…): 100% 6.36M/6.36M [00:00<00:00, 18.1MB/s]
wikitext-2-raw-v1/validation-00000-of-00(…): 100% 657k/657k [00:00<00:00, 3.29MB/s]
Generating test split: 100% 4358/4358 [00:00<00:00, 116527.76 examples/s]
Generating train split: 100% 36718/36718 [00:00<00:00, 777464.84 examples/s]
Generating validation split: 100% 3760/3

In [ ]:
import os
import subprocess
import re
import json
import pandas as pd

# ============================================================================
# 1. SETUP (TANPA DOWNGRADE DATASETS)
# ============================================================================
print(" Menyiapkan environment (menggunakan default Colab)...")
# Hapus instalasi datasets==2.16.1 karena itu yang menyebabkan error dataclass
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"

eval_dir = "/content/BitDistiller/test/general"
model_path = "../../models/tinyllama_2bit_bitdistiller"
os.makedirs("/content/results", exist_ok=True)

# Gunakan task yang stabil
tasks = "hellaswag,winogrande,arc_easy"
print("✅ Environment siap.\n")

# ============================================================================
# 2. FUNGSI EVALUASI OTOMATIS
# ============================================================================
def run_evaluation():
    results = {"model": "BitDistiller 2-bit (TinyLlama)"}

    # --- A. Evaluasi Perplexity ---
    print("📊 [1/2] Menghitung WikiText-2 Perplexity...")
    cmd_ppl = ["python", "wiki_ppl.py", "--model", model_path, "--quant_type", "int", "--bits", "2", "--group_size", "128"]
    proc_ppl = subprocess.run(cmd_ppl, cwd=eval_dir, capture_output=True, text=True)

    ppl_match = re.search(r"ppl:\s*\n?\s*([\d.]+)", proc_ppl.stdout + proc_ppl.stderr)
    if ppl_match:
        results["wikitext2_ppl"] = round(float(ppl_match.group(1)), 4)
        print(f"   ✅ PPL: {results['wikitext2_ppl']}")
    else:
        results["wikitext2_ppl"] = "Error"
        print("   ❌ Gagal mengekstrak PPL.")

    # --- B. Evaluasi Zero-Shot QA ---
    print(f"\n [2/2] Menghitung Zero-shot QA Accuracy ({tasks})...")
    cmd_qa = ["python", "llm_eval.py", "--model", model_path, "--eval_tasks", tasks, "--test_set", "--bits", "2", "--group_size", "128", "--quant_type", "int", "--num_fewshot", "0", "--batch_size", "2"]

    proc_qa = subprocess.run(cmd_qa, cwd=eval_dir, capture_output=True, text=True)

    # Regex untuk menangkap dictionary hasil {'results': {'hellaswag': {'acc': ...}}}
    #dict_match = re.search(r"\{'results':.*?\}", proc_qa.stdout + proc_qa.stderr)
    dict_match = re.search(r"\{'results':.*\}", proc_qa.stdout + proc_qa.stderr, re.DOTALL)

    if dict_match:
        try:
            import ast
            parsed = ast.literal_eval(dict_match.group(0))
            for task in tasks.split(","):
                if task in parsed['results']:
                    metrics = parsed['results'][task]
                    acc = metrics.get('acc_norm', metrics.get('acc', 0))
                    results[f"QA_{task}_acc_norm"] = round(acc * 100, 2)
            print(f"   ✅ QA Results berhasil diambil dari sistem: { {k:v for k,v in results.items() if 'QA' in k} }")
        except Exception as e:
            print(f"   ❌ Gagal parse QA output: {e}")
    else:
        print("   ❌ Tidak menemukan dictionary hasil QA. Cek error di bawah:")
        # Tampilkan 300 karakter terakhir untuk debugging
        print((proc_qa.stdout + proc_qa.stderr)[-300:])

    return results

# ============================================================================
# 3. JALANKAN DAN SIMPAN HASIL
# ============================================================================
print("="*80)
print("🚀 MEMULAI EVALUASI OTOMATIS BITDISTILLER 2-BIT")
print("="*80)

final_results = run_evaluation()

# Tampilkan sebagai DataFrame
df = pd.DataFrame([final_results]).T
df.columns = ["Value"]

print("\n" + "="*80)
print("📊 HASIL EVALUASI FINAL (Diambil Otomatis dari Sistem)")
print("="*80)
print(df.to_markdown())
print("="*80)

# Simpan ke JSON dan CSV
with open("/content/results/bitdistiller_2bit_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

df.to_csv("/content/results/bitdistiller_2bit_results.csv")

print("\n💾 Hasil disimpan ke:")
print("   - /content/results/bitdistiller_2bit_results.json")
print("   - /content/results/bitdistiller_2bit_results.csv")

from google.colab import files
files.download("/content/results/bitdistiller_2bit_results.csv")

 Menyiapkan environment (menggunakan default Colab)...
✅ Environment siap.

🚀 MEMULAI EVALUASI OTOMATIS BITDISTILLER 2-BIT
📊 [1/2] Menghitung WikiText-2 Perplexity...
   ✅ PPL: 23.7603

 [2/2] Menghitung Zero-shot QA Accuracy (hellaswag,winogrande,arc_easy)...
   ❌ Gagal parse QA output: malformed node or string on line 1: <ast.Call object at 0x7f0f2284d910>

📊 HASIL EVALUASI FINAL (Diambil Otomatis dari Sistem)
|               | Value                          |
|:--------------|:-------------------------------|
| model         | BitDistiller 2-bit (TinyLlama) |
| wikitext2_ppl | 23.7603                        |

💾 Hasil disimpan ke:
   - /content/results/bitdistiller_2bit_results.json
   - /content/results/bitdistiller_2bit_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import subprocess
import re
import json
import pandas as pd

# ============================================================================
# 1. SETUP
# ============================================================================
print("🔧 Menyiapkan environment...")
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"

eval_dir = "/content/BitDistiller/test/general"
model_path = "../../models/tinyllama_2bit_bitdistiller"
os.makedirs("/content/results", exist_ok=True)

# Gunakan task yang stabil
tasks = "hellaswag,winogrande,arc_easy"
print("✅ Environment siap.\n")

# ============================================================================
# 2. FUNGSI EVALUASI OTOMATIS (DENGAN PARSING BRACE COUNTING)
# ============================================================================
def run_evaluation():
    results = {"model": "BitDistiller 2-bit (TinyLlama)"}

    # --- A. Evaluasi Perplexity ---
    print("📊 [1/2] Menghitung WikiText-2 Perplexity...")
    cmd_ppl = ["python", "wiki_ppl.py", "--model", model_path, "--quant_type", "int", "--bits", "2", "--group_size", "128"]
    proc_ppl = subprocess.run(cmd_ppl, cwd=eval_dir, capture_output=True, text=True)

    ppl_match = re.search(r"ppl:\s*\n?\s*([\d.]+)", proc_ppl.stdout + proc_ppl.stderr)
    if ppl_match:
        results["wikitext2_ppl"] = round(float(ppl_match.group(1)), 4)
        print(f"   ✅ PPL: {results['wikitext2_ppl']}")
    else:
        results["wikitext2_ppl"] = "Error"
        print("   ❌ Gagal mengekstrak PPL.")

    # --- B. Evaluasi Zero-Shot QA ---
    print(f"\n📊 [2/2] Menghitung Zero-shot QA Accuracy ({tasks})...")
    cmd_qa = ["python", "llm_eval.py", "--model", model_path, "--eval_tasks", tasks, "--test_set", "--bits", "2", "--group_size", "128", "--quant_type", "int", "--num_fewshot", "0", "--batch_size", "2"]

    proc_qa = subprocess.run(cmd_qa, cwd=eval_dir, capture_output=True, text=True)
    text_output = proc_qa.stdout + proc_qa.stderr

    # CARA AMAN: Cari {'results': dan hitung kurung kurawal hingga tertutup sempurna
    start_idx = text_output.find("{'results':")
    qa_parsed = False

    if start_idx != -1:
        brace_count = 0
        end_idx = -1
        # Iterasi dari posisi {'results': untuk menemukan kurung tutup yang pas
        for i in range(start_idx, len(text_output)):
            if text_output[i] == '{':
                brace_count += 1
            elif text_output[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    end_idx = i + 1
                    break

        if end_idx != -1:
            dict_str = text_output[start_idx:end_idx]
            try:
                import ast
                parsed = ast.literal_eval(dict_str)
                for task in tasks.split(","):
                    if task in parsed['results']:
                        metrics = parsed['results'][task]
                        acc = metrics.get('acc_norm', metrics.get('acc', 0))
                        results[f"QA_{task}_acc_norm"] = round(acc * 100, 2)
                print(f"   ✅ QA Results berhasil diambil: { {k:v for k,v in results.items() if 'QA' in k} }")
                qa_parsed = True
            except Exception as e:
                print(f"    Gagal parse QA output (isi dictionary korup): {e}")
        else:
            print("   ❌ Kurung kurawal tidak tertutup sempurna di output.")
    else:
        print("   ❌ String {'results': tidak ditemukan di output.")

    if not qa_parsed:
        print("\n--- DEBUG: 500 karakter terakhir dari output ---")
        print(text_output[-500:])

    return results

# ============================================================================
# 3. JALANKAN DAN SIMPAN HASIL
# ============================================================================
print("="*80)
print("🚀 MEMULAI EVALUASI OTOMATIS BITDISTILLER 2-BIT")
print("="*80)

final_results = run_evaluation()

# Tampilkan sebagai DataFrame
df = pd.DataFrame([final_results]).T
df.columns = ["Value"]

print("\n" + "="*80)
print("📊 HASIL EVALUASI FINAL (Diambil Otomatis dari Sistem)")
print("="*80)
print(df.to_markdown())
print("="*80)

# Simpan ke JSON dan CSV
with open("/content/results/bitdistiller_2bit_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

df.to_csv("/content/results/bitdistiller_2bit_results.csv")

print("\n💾 Hasil disimpan ke:")
print("   - /content/results/bitdistiller_2bit_results.json")
print("   - /content/results/bitdistiller_2bit_results.csv")

from google.colab import files
files.download("/content/results/bitdistiller_2bit_results.csv")

🔧 Menyiapkan environment...
✅ Environment siap.

🚀 MEMULAI EVALUASI OTOMATIS BITDISTILLER 2-BIT
📊 [1/2] Menghitung WikiText-2 Perplexity...
   ✅ PPL: 23.7603

📊 [2/2] Menghitung Zero-shot QA Accuracy (hellaswag,winogrande,arc_easy)...
    Gagal parse QA output (isi dictionary korup): malformed node or string on line 1: <ast.Call object at 0x7f0f22896890>

--- DEBUG: 500 karakter terakhir dari output ---
0, 71.06it/s]
100%|██████████| 52175/52175 [14:05<00:00, 61.70it/s]


📊 HASIL EVALUASI FINAL (Diambil Otomatis dari Sistem)
|               | Value                          |
|:--------------|:-------------------------------|
| model         | BitDistiller 2-bit (TinyLlama) |
| wikitext2_ppl | 23.7603                        |

💾 Hasil disimpan ke:
   - /content/results/bitdistiller_2bit_results.json
   - /content/results/bitdistiller_2bit_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>